# Module 1 · — Foundational Infrastructure & Environment Orchestration

**Prerequisites:** NB01 (Python Primer)  
**Difficulty:** Intermediate  
**Estimated Time:** 60-90 minutes

**Architectural Layer:** Infrastructure Foundation  
**Toolchain:** Linux CLI · Git · Docker · Kubernetes · Kubeflow  
**Objective:** Master the computational infrastructure that ALL MLOps systems are built upon — from terminal commands to container orchestration.

---

## 🧠 Why Infrastructure Before Machine Learning?

### The Reality Check

Most ML courses start with `model.fit()`. Most ML FAILURES happen because of infrastructure:
- The model works on a laptop but crashes on the server
- The training run dies at hour 47 with no logs to debug
- The deployed container eats all GPU memory on a shared cluster
- A teammate's experiment broke yours because they upgraded a library

**Real-world scenario:**  
A data scientist spends 3 weeks perfecting a model. On deployment day, it doesn't work because the server has Python 3.9 (not 3.11), PyTorch 2.1 (not 2.3), and a completely different CUDA version. Three weeks wasted. This notebook teaches you to PREVENT this.

### 🤔 Why Linux and Not Windows?

| Environment | Where ML Models Are Trained | Where ML Models Are Deployed |
|------------|---------------------------|-----------------------------|
| Windows | ❌ Rare (driver issues, path problems) | ❌ Never in production |
| macOS | ✅ Development only | ❌ No NVIDIA GPU support |
| Linux | ✅ Standard for training (Ubuntu/RHEL) | ✅ ALL production servers |

**100% of cloud GPU instances (AWS, GCP, Azure) run Linux.** If you can't navigate a Linux terminal, you can't debug production ML systems.

### What This Notebook Covers

```
1. Linux Terminal → Parse ML training logs with grep/awk/sed
2. Git → Version control EVERYTHING (code, configs, pipelines)
3. Docker → Package models into reproducible containers
4. Kubernetes → Orchestrate containers at scale with GPU allocation
5. Kubeflow → ML-specific orchestration on top of Kubernetes
```

---

## 📑 Table of Contents

* [🧠 Why Infrastructure Before Machine Learning?](#why-infrastructure-before-machine-learning)
  * [The Reality Check](#the-reality-check)
  * [🤔 Why Linux and Not Windows?](#why-linux-and-not-windows)
  * [What This Notebook Covers](#what-this-notebook-covers)
  * [🎓 Junior to Senior: Concept Bridge](#junior-to-senior-concept-bridge)
* [1 · Linux Terminal Mastery for ML Engineers](#1-linux-terminal-mastery-for-ml-engineers)
  * [🤔 Why Do I Need Terminal Skills?](#why-do-i-need-terminal-skills)
  * [Key Commands Every ML Engineer Must Know](#key-commands-every-ml-engineer-must-know)
* [2 · Git: Version Control for Everything](#2-git-version-control-for-everything)
  * [🤔 Why Is Git the Foundation of MLOps?](#why-is-git-the-foundation-of-mlops)
  * [⚖️ Git Branching Strategy for ML Teams](#git-branching-strategy-for-ml-teams)
* [3 · Docker: Reproducible ML Environments](#3-docker-reproducible-ml-environments)
  * [🤔 What Problem Does Docker Solve?](#what-problem-does-docker-solve)
  * [🤔 What is a Multi-Stage Dockerfile?](#what-is-a-multi-stage-dockerfile)
* [4 · Kubernetes: Orchestrating ML at Scale](#4-kubernetes-orchestrating-ml-at-scale)
  * [🤔 Why Can't We Just Use Docker Alone?](#why-cant-we-just-use-docker-alone)
  * [Kubernetes Core Concepts](#kubernetes-core-concepts)
  * [🤔 What Are Resource Limits and Why Are They Critical?](#what-are-resource-limits-and-why-are-they-critical)
* [5 · Kubeflow: ML-Specific Orchestration](#5-kubeflow-ml-specific-orchestration)
  * [🤔 Why Kubeflow When We Already Have Kubernetes?](#why-kubeflow-when-we-already-have-kubernetes)
  * [⚖️ Trade-offs: Kubeflow vs Other Orchestrators](#trade-offs-kubeflow-vs-other-orchestrators)
* [6 - GitHub Actions: CI/CD for ML Projects](#6-github-actions-cicd-for-ml-projects)
  * [Why CI/CD Matters for ML](#why-cicd-matters-for-ml)
  * [Key Concepts](#key-concepts)
  * [.dockerignore: Keep Your Images Clean](#dockerignore-keep-your-images-clean)
  * [Docker Compose: Multi-Container ML Systems](#docker-compose-multi-container-ml-systems)
* [✅ Knowledge Check](#knowledge-check)
  * [Question 1: Kubernetes Limits](#question-1-kubernetes-limits)
  * [Question 2: Docker Multi-Stage Builds](#question-2-docker-multi-stage-builds)
  * [Question 3: Git Operations](#question-3-git-operations)
* [🔨 Practical Practice](#practical-practice)
  * [Exercise 1: Advanced Grep](#exercise-1-advanced-grep)
  * [Exercise 2: Python to Dockerfile translation](#exercise-2-python-to-dockerfile-translation)
  * [Exercise 3: Kubernetes Scaling](#exercise-3-kubernetes-scaling)
* [🎯 Summary & Key Takeaways](#summary-key-takeaways)
  * [🧠 Key Questions](#key-questions)


### 🎓 Junior to Senior: Concept Bridge

**The Senior Perspective:** AI code is useless if it only runs on your laptop. Seniors view Docker and Kubernetes as the *actual product delivery mechanism*, not just DevOps chores. Reproducibility is the foundation of MLOps.

**Mental Model:** Docker is a self-contained shipping container (your code + OS + dependencies). Kubernetes is the massive port authority that manages thousands of containers, deciding where they go and replacing them if they sink.

**Common Junior Pitfall:** Hardcoding file paths (like `C:/Users/...`) in ML scripts instead of using environment variables and relative Docker paths, causing deployment failures on the server.

---


## 1 · Linux Terminal Mastery for ML Engineers

### 🤔 Why Do I Need Terminal Skills?

Because production ML servers have NO graphical interface. When your model crashes at 3 AM, you SSH into the server and debug with the terminal. Period.

### Key Commands Every ML Engineer Must Know

| Command | What It Does | ML Use Case |
|---------|-------------|-------------|
| `grep` | Search text patterns in files | Find errors in training logs |
| `awk` | Process and transform structured text | Extract metrics from logs |
| `sed` | Stream-edit text (find & replace) | Fix config files in-place |
| `tail -f` | Watch a file in real-time | Monitor training progress live |
| `top` / `nvidia-smi` | System resource monitoring | Check GPU/CPU/Memory usage |
| `ssh` | Remote server access | Connect to training servers |
| `scp` / `rsync` | Copy files between machines | Transfer model weights |
| `chmod` / `chown` | File permissions | Fix "permission denied" errors |

In [ ]:
# Cell 1 — Generate a realistic ML training log
#
# In real life, training frameworks (PyTorch, TensorFlow) write logs like this.
# When something goes wrong, you need to parse these logs to find the error.
# We'll simulate common ML failure scenarios and then use Linux tools to find them.

import datetime, random

log_lines = []
random.seed(42)

for epoch in range(1, 11):
    for step in range(1, 51):
        ts = datetime.datetime(2026, 2, 28, 10, 0, 0) + datetime.timedelta(seconds=epoch*50+step)
        loss = 2.5 / (epoch * 0.8 + step * 0.01) + random.uniform(-0.05, 0.05)
        lr = 2e-4 * (0.95 ** epoch)
        gpu_mem = 14.2 + random.uniform(0, 2.0)
        log_lines.append(f"[{ts.isoformat()}] INFO  Epoch {epoch:02d} Step {step:03d} | loss={loss:.4f} lr={lr:.6f} gpu_mem={gpu_mem:.1f}GB")

        # Inject realistic problems
        if epoch == 3 and step == 25:
            log_lines.append(f"[{ts.isoformat()}] WARNING  GPU memory approaching limit: 15.8/16.0 GB")
        if epoch == 5 and step == 30:
            log_lines.append(f"[{ts.isoformat()}] ERROR  CUDA out of memory. Tried to allocate 512 MiB (GPU 0; 16.0 GiB total; 15.4 GiB already allocated)")
        if epoch == 7 and step == 10:
            log_lines.append(f"[{ts.isoformat()}] WARNING  Loss diverging: NaN detected at step {step}")
        if epoch == 7 and step == 11:
            log_lines.append(f"[{ts.isoformat()}] ERROR  RuntimeError: Loss is NaN. Gradient explosion suspected. Check learning rate.")
        if epoch == 9 and step == 40:
            log_lines.append(f"[{ts.isoformat()}] ERROR  ModuleNotFoundError: No module named 'bitsandbytes'. Install: pip install bitsandbytes")

with open("training.log", "w") as f:
    f.write("\n".join(log_lines))

print(f"📄 Generated training.log: {len(log_lines)} lines")
print(f"   Contains: OOM error, NaN loss, missing dependency")

In [ ]:
%%bash
# Cell 2 — grep: Find ALL errors in the training log
#
# grep = "Global Regular Expression Print"
# It searches through text line-by-line and prints matching lines.
#
# Flags:
#   -n  = show line numbers (critical for debugging!)
#   -i  = case-insensitive
#   -c  = count matches instead of showing them
#
# 🤔 Why is this better than opening the file in a text editor?
# Because training logs can be 500,000+ lines. grep finds needles in haystacks.

echo "=== 🔴 ALL ERRORS ==="
grep -n "ERROR" training.log

echo ""
echo "=== ⚠️ ALL WARNINGS ==="
grep -n "WARNING" training.log

echo ""
echo "=== 📊 Error/Warning counts ==="
echo "Errors: $(grep -c 'ERROR' training.log)"
echo "Warnings: $(grep -c 'WARNING' training.log)"

In [ ]:
%%bash
# Cell 3 — awk: Extract and analyze training metrics
#
# awk processes text column by column.
# It's like pandas for the terminal — but it works on files of ANY size
# without loading them into memory.
#
# This command:
# 1. Finds lines containing 'loss='
# 2. Extracts the loss value
# 3. Prints the epoch, step, and loss
#
# 🤔 When to use awk vs Python?
# - awk: Quick one-liners, huge files (>1GB), remote servers
# - Python: Complex analysis, visualization, multi-step processing

echo "=== First 10 training steps (Epoch, Step, Loss) ==="
grep 'loss=' training.log | head -10 | awk -F'[ |=]' '{for(i=1;i<=NF;i++){if($i=="loss")print $(i-4), $(i-2), $(i+1)}}'

echo ""
echo "=== GPU memory usage (lines with gpu_mem > 15.5 GB) ==="
grep 'gpu_mem=' training.log | awk -F'gpu_mem=' '{split($2,a,"GB"); if(a[1]+0 > 15.5) print $0}' | head -5

In [ ]:
%%bash
# Cell 4 — sed: Fix configuration files in-place
#
# sed = "Stream Editor". It modifies text WITHOUT opening a text editor.
# Essential for automated scripts and CI/CD pipelines.
#
# Real-world scenario:
# Your training config has learning_rate: 0.001 but you need 0.0001.
# On a remote server with no GUI, you use sed.

# Create a sample config file
cat > training_config.yaml << 'EOF'
model:
  name: microsoft/phi-2
  learning_rate: 0.001
  batch_size: 16
  epochs: 10
  optimizer: adamw
data:
  path: /data/raw/train.csv
  validation_split: 0.2
EOF

echo "=== BEFORE ==="
cat training_config.yaml

# Fix the learning rate
sed -i 's/learning_rate: 0.001/learning_rate: 0.0001/' training_config.yaml
# Fix the batch size (reduce to prevent OOM)
sed -i 's/batch_size: 16/batch_size: 4/' training_config.yaml

echo ""
echo "=== AFTER (learning_rate and batch_size fixed) ==="
cat training_config.yaml

---

## 2 · Git: Version Control for Everything

### 🤔 Why Is Git the Foundation of MLOps?

In MLOps, **EVERYTHING** must be version-controlled:

| What | Why |
|------|-----|
| Training code | Reproduce any experiment |
| Config files | Know exactly what hyperparameters were used |
| Dockerfiles | Reproduce the exact deployment environment |
| K8s manifests | Rollback infrastructure changes |
| Pipeline definitions | Audit workflow changes |
| Data schemas | Track data contract evolution |

**🤔 What about model weights and data?**  
Git is NOT designed for large binary files. For models and data, use:
- **DVC** (Data Version Control) — Git-like interface for large files (Notebook 01)
- **lakeFS** — Git-like branches for data lakes (Notebook 01)
- **Git LFS** — Git Large File Storage (simpler but less powerful)

In [ ]:
%%bash
# Cell 5 — Simulate a Git workflow for ML projects
#
# This simulates the standard MLOps Git workflow:
# 1. main branch = production (always stable)
# 2. feature branches = experiments (isolated)
# 3. merge = only after code review + tests pass

mkdir -p /tmp/mlops-git-demo && cd /tmp/mlops-git-demo
git init -q
git config user.email "ml-engineer@company.com"
git config user.name "ML Engineer"

# Initial commit: baseline model
echo 'learning_rate: 0.001' > config.yaml
echo 'import sklearn' > train.py
git add -A && git commit -q -m "feat: initial training pipeline"

# Create experiment branch
git checkout -q -b experiment/lower-lr
echo 'learning_rate: 0.0001' > config.yaml
git add -A && git commit -q -m "experiment: reduce LR to 0.0001"

# Show the divergence
echo "=== Git Log (all branches) ==="
git log --oneline --all --graph

echo ""
echo "=== Diff between main and experiment ==="
git diff main..experiment/lower-lr -- config.yaml

### ⚖️ Git Branching Strategy for ML Teams

```
main (production)   ──●──────────●──────────────●──── (stable releases)
                       \        /                \
staging              ──●──●──●──●                 ●── (testing)
                       \    /
experiment/new-lr    ──●──●                           (data scientist A)
experiment/larger-model  ──●──●──●                    (data scientist B)
```

**Rule:** Never push directly to `main`. Always use pull requests with code review and passing CI/CD tests.

---

## 3 · Docker: Reproducible ML Environments

### 🤔 What Problem Does Docker Solve?

**"It works on my machine!"** — the most famous lie in software engineering.

Docker creates a **container** — a lightweight, portable package that bundles your code + dependencies + runtime into a single unit that runs IDENTICALLY everywhere.

| Without Docker | With Docker |
|---------------|-------------|
| "My laptop has PyTorch 2.3, server has 2.1" | Same PyTorch version everywhere |
| "It needs CUDA 12.4 but server has 11.8" | CUDA version baked into image |
| "I forgot to install the `sentencepiece` package" | All deps declared and installed |
| "Which Python version was that model trained with?" | Python version is explicit |

### 🤔 What is a Multi-Stage Dockerfile?

A multi-stage build uses TWO images:
1. **Build stage**: Large image with compilers, build tools (gcc, cmake) — used to install packages
2. **Runtime stage**: Minimal image with ONLY what's needed to run the model

| Approach | Image Size | Security |
|----------|-----------|----------|
| Single-stage | ~8 GB | ❌ Contains compilers, build tools (attack surface) |
| Multi-stage | ~3 GB | ✅ Minimal runtime only |

**🤔 Why does image size matter?**
- Faster deployment (download from registry)
- Lower storage costs in container registries
- Smaller attack surface (fewer packages = fewer vulnerabilities)

In [ ]:
# Cell 6 — Generate an optimized multi-stage Dockerfile for ML
#
# This Dockerfile follows production best practices:
# 1. Multi-stage: build deps in stage 1, copy only runtime to stage 2
# 2. Layer caching: requirements.txt BEFORE code (rebuild only when deps change)
# 3. Non-root user: security best practice
# 4. Health check: K8s/Docker can detect if the service is alive
#
# 🤔 Why copy requirements.txt BEFORE the rest of the code?
# Docker caches each layer. If requirements.txt hasn't changed,
# Docker reuses the cached layer instead of reinstalling all packages.
# This turns a 10-minute build into a 30-second build.

dockerfile_content = '''# ============================================================
# STAGE 1: Build — install Python packages with build tools
# ============================================================
FROM nvidia/cuda:12.4.0-devel-ubuntu22.04 AS builder

# Prevent interactive prompts during apt-get
ENV DEBIAN_FRONTEND=noninteractive

# Install Python and build dependencies
RUN apt-get update && apt-get install -y --no-install-recommends \\
    python3.11 python3.11-venv python3-pip \\
    build-essential gcc g++ cmake \\
    && rm -rf /var/lib/apt/lists/*

# Create virtual environment (isolation inside the container too!)
RUN python3.11 -m venv /opt/venv
ENV PATH="/opt/venv/bin:$PATH"

# Install Python dependencies FIRST (layer caching)
COPY requirements.txt /tmp/requirements.txt
RUN pip install --no-cache-dir -r /tmp/requirements.txt

# ============================================================
# STAGE 2: Runtime — minimal image for serving
# ============================================================
FROM nvidia/cuda:12.4.0-runtime-ubuntu22.04

ENV DEBIAN_FRONTEND=noninteractive

# Install ONLY runtime dependencies (no compilers!)
RUN apt-get update && apt-get install -y --no-install-recommends \\
    python3.11 python3.11-venv \\
    && rm -rf /var/lib/apt/lists/*

# Copy the virtual environment from builder stage
COPY --from=builder /opt/venv /opt/venv
ENV PATH="/opt/venv/bin:$PATH"

# Create non-root user (SECURITY: never run as root in production)
RUN useradd -m -s /bin/bash mluser
USER mluser
WORKDIR /home/mluser/app

# Copy application code
COPY --chown=mluser:mluser . .

# Health check: K8s uses this to know if the container is alive
HEALTHCHECK --interval=30s --timeout=10s --retries=3 \\
    CMD python3.11 -c "import requests; requests.get('http://localhost:8000/health')" || exit 1

# Expose the serving port
EXPOSE 8000

# Start the inference server
CMD ["python3.11", "-m", "uvicorn", "service:app", "--host", "0.0.0.0", "--port", "8000"]
'''

with open("Dockerfile", "w") as f:
    f.write(dockerfile_content)

print("🐳 Multi-stage Dockerfile generated")
print("\n📋 Stage comparison:")
print("   Builder (nvidia/cuda:12.4.0-devel):  ~8.5 GB (has compilers)")
print("   Runtime (nvidia/cuda:12.4.0-runtime): ~3.2 GB (minimal)")
print("   Savings: ~5.3 GB per image!")
print("\n🔒 Security features:")
print("   ✅ Non-root user (mluser)")
print("   ✅ No build tools in runtime image")
print("   ✅ Health check for container orchestration")

In [ ]:
# Cell 7 — Generate requirements.txt for the ML service

requirements = """# ML Inference Dependencies
torch==2.3.0
transformers==4.41.0
accelerate==0.30.0
vllm==0.5.0
fastapi==0.111.0
uvicorn==0.30.0
pydantic==2.7.0
numpy==1.26.4
requests==2.32.0
"""

with open("requirements.txt", "w") as f:
    f.write(requirements)

print("📋 requirements.txt generated")
print("\n🤔 Why pin EXACT versions (torch==2.3.0 not torch>=2.0)?")
print("   Because 'pip install torch' today gives 2.3.0,")
print("   but in 3 months it gives 2.5.0 — which might break your model.")
print("   Pinned versions = reproducible builds.")

In [ ]:
%%bash
# Cell 8 — Docker build commands (explanation, not execution)
#
# These are the commands you'd run in a real environment.
# We print them here for reference.

echo "=== Docker Build & Push Workflow ==="
echo ""
echo "# Build the image (uses Dockerfile in current directory)"
echo "docker build -t registry.company.com/ml-service:v1.0 ."
echo ""
echo "# Test locally before pushing"
echo "docker run --gpus all -p 8000:8000 registry.company.com/ml-service:v1.0"
echo ""
echo "# Scan for vulnerabilities (NEVER skip this!)"
echo "trivy image registry.company.com/ml-service:v1.0"
echo ""
echo "# Push to container registry (only after scan passes)"
echo "docker push registry.company.com/ml-service:v1.0"
echo ""
echo "# Useful debugging commands:"
echo "docker logs <container_id>          # View container output"
echo "docker exec -it <container_id> bash  # Shell into running container"
echo "docker images --format 'table {{.Repository}}\t{{.Size}}'  # Check image sizes"

---

## 4 · Kubernetes: Orchestrating ML at Scale

### 🤔 Why Can't We Just Use Docker Alone?

Docker runs ONE container on ONE machine. In production, you need:

| Challenge | Docker Alone | Docker + Kubernetes |
|-----------|-------------|--------------------|
| Run 10 copies for high traffic | Manually start on 10 servers | `replicas: 10` (one line) |
| Container crashes | It stays dead | Auto-restarts within seconds |
| Deploy new version | Stop old, start new (downtime) | Rolling update (zero downtime) |
| Scale during peak hours | Manually add servers | Auto-scale based on CPU/GPU |
| GPU allocation | Manually assign GPUs | Declarative GPU scheduling |

### Kubernetes Core Concepts

| Concept | What It Is | Analogy |
|---------|-----------|--------|
| **Pod** | Smallest unit — 1+ containers running together | A single worker |
| **Deployment** | Manages a group of identical pods | A team of workers |
| **Service** | Stable network endpoint to reach pods | The reception desk |
| **Namespace** | Isolated virtual cluster | Different departments |
| **Node** | Physical/virtual machine in the cluster | A desk in the office |

### 🤔 What Are Resource Limits and Why Are They Critical?

Without resource limits, a single model container can:
- Consume ALL available GPU memory → other models crash
- Use ALL CPU cores → API gateway becomes unresponsive
- Fill ALL disk space with checkpoints → entire node goes down

**Real-world scenario:**  
A team deployed a model without memory limits. During a traffic spike, it consumed 64 GB of RAM (the entire node). Every other service on that node — including the monitoring system — crashed. Nobody knew what happened because the monitoring was dead too.

In [ ]:
# Cell 9 — Generate Kubernetes deployment with GPU resources
#
# This manifest tells Kubernetes:
# "Run 3 copies of my ML container, each with exactly 1 GPU,
#  max 16GB memory, and restart it if it crashes."
#
# KEY CONCEPTS:
# - requests: MINIMUM resources guaranteed to the pod
# - limits: MAXIMUM resources the pod can ever use
#   If a pod exceeds memory limits → it gets killed (OOMKilled)
#   If a pod exceeds CPU limits → it gets throttled (slowed down)
#
# - nvidia.com/gpu: special resource type for GPU allocation
#   Each GPU is atomic — you can't request 0.5 GPUs
#   (unless using Multi-Instance GPU — MIG — on A100/H100)
#
# - tolerations: allows this pod to be scheduled on GPU nodes
#   GPU nodes have a "taint" that prevents non-GPU pods from landing there
#   (so expensive GPU nodes aren't wasted on lightweight tasks)

import yaml

k8s_deployment = {
    "apiVersion": "apps/v1",
    "kind": "Deployment",
    "metadata": {
        "name": "ml-inference-service",
        "namespace": "ml-production",
        "labels": {"app": "ml-inference", "version": "v1.0", "team": "ml-platform"},
    },
    "spec": {
        "replicas": 3,  # 3 copies for high availability
        "selector": {"matchLabels": {"app": "ml-inference"}},
        "strategy": {
            "type": "RollingUpdate",  # Zero-downtime deployment
            "rollingUpdate": {
                "maxSurge": 1,         # At most 1 extra pod during update
                "maxUnavailable": 0,   # Never have 0 running pods
            },
        },
        "template": {
            "metadata": {"labels": {"app": "ml-inference", "version": "v1.0"}},
            "spec": {
                "containers": [{
                    "name": "inference-server",
                    "image": "registry.company.com/ml-service:v1.0",
                    "ports": [{"containerPort": 8000, "name": "http"}],
                    "resources": {
                        "requests": {
                            "cpu": "4",
                            "memory": "8Gi",
                            "nvidia.com/gpu": "1",  # Guarantee 1 GPU
                        },
                        "limits": {
                            "cpu": "8",
                            "memory": "16Gi",
                            "nvidia.com/gpu": "1",  # Cannot use more than 1 GPU
                        },
                    },
                    "env": [
                        {"name": "MODEL_NAME", "value": "microsoft/phi-2"},
                        {"name": "CUDA_VISIBLE_DEVICES", "value": "0"},
                    ],
                    "readinessProbe": {
                        "httpGet": {"path": "/health", "port": 8000},
                        "initialDelaySeconds": 60,  # Wait 60s for model loading
                        "periodSeconds": 10,
                        "failureThreshold": 3,
                    },
                    "livenessProbe": {
                        "httpGet": {"path": "/health", "port": 8000},
                        "initialDelaySeconds": 120,
                        "periodSeconds": 30,
                    },
                }],
                "tolerations": [
                    {"key": "nvidia.com/gpu", "operator": "Exists", "effect": "NoSchedule"},
                ],
                "nodeSelector": {"accelerator": "nvidia-gpu"},
            },
        },
    },
}

manifest_yaml = yaml.dump(k8s_deployment, default_flow_style=False)
with open("k8s_ml_deployment.yaml", "w") as f:
    f.write(manifest_yaml)

print("☸️ Kubernetes Deployment Manifest:")
print(manifest_yaml)

In [ ]:
# Cell 10 — Explain kubectl commands for ML engineers

kubectl_cheatsheet = """
☸️ Essential kubectl Commands for ML Engineers
═══════════════════════════════════════════════

# Deploy / Update
kubectl apply -f k8s_ml_deployment.yaml          # Create or update deployment
kubectl rollout status deployment/ml-inference     # Watch deployment progress
kubectl rollout undo deployment/ml-inference       # ROLLBACK to previous version

# Inspect
kubectl get pods -n ml-production                  # List all pods
kubectl describe pod <pod-name>                    # Detailed pod info (events, errors)
kubectl logs <pod-name> --tail=100                 # Last 100 log lines
kubectl logs <pod-name> -f                         # Stream logs in real-time

# Debug
kubectl exec -it <pod-name> -- /bin/bash           # Shell into a running pod
kubectl top pods -n ml-production                  # CPU/memory usage per pod
kubectl get events --sort-by='.lastTimestamp'      # Recent cluster events

# Scale
kubectl scale deployment/ml-inference --replicas=5 # Manual scale to 5 pods

# GPU-specific
kubectl describe nodes | grep -A5 'nvidia.com/gpu' # Check GPU availability
"""

print(kubectl_cheatsheet)

---

## 5 · Kubeflow: ML-Specific Orchestration

### 🤔 Why Kubeflow When We Already Have Kubernetes?

Kubernetes is a GENERAL container orchestrator. It knows nothing about ML. Kubeflow adds ML-specific features on top:

| Feature | Raw Kubernetes | Kubeflow |
|---------|---------------|----------|
| ML pipelines (DAGs) | Not supported | Built-in (KFP) |
| Jupyter notebooks | Not supported | Notebook servers as pods |
| Hyperparameter tuning | Not supported | Katib (automated search) |
| Model serving | Manual configuration | KServe (built-in) |
| Multi-user isolation | Basic RBAC | Full profile/namespace isolation |

### ⚖️ Trade-offs: Kubeflow vs Other Orchestrators

| Tool | Complexity | Best For | Learning Curve |
|------|-----------|----------|----------------|
| **Kubeflow** | High | Large teams, K8s-first organizations | Steep |
| **Metaflow** | Low | Data scientists, quick prototyping | Easy |
| **Airflow** | Medium | Data engineering, ETL scheduling | Moderate |
| **Prefect** | Low-Medium | Dynamic workflows, observability | Easy |

**🤔 When NOT to use Kubeflow?**
- Small team (< 5 engineers): Too much overhead
- No existing K8s cluster: Building one just for Kubeflow is expensive
- Quick experiments: Use Metaflow or simple scripts instead

In [ ]:
# Cell 11 — Kubeflow architecture overview

kubeflow_architecture = """
🔧 Kubeflow Architecture on Kubernetes
══════════════════════════════════════

┌─────────────────────────────────────────────────┐
│                 Kubeflow Dashboard               │  ← Web UI for managing everything
├─────────────────────────────────────────────────┤
│  Kubeflow Pipelines (KFP)                       │  ← ML workflow DAGs (Notebook 06)
│  Jupyter Notebook Servers                        │  ← On-demand notebook environments
│  Katib (Hyperparameter Tuning)                   │  ← Automated search (Bayesian, NAS)
│  KServe (Model Serving)                          │  ← Inference endpoints
│  Training Operators (TFJob, PyTorchJob)           │  ← Distributed training
├─────────────────────────────────────────────────┤
│  Istio Service Mesh                              │  ← Traffic routing, security
│  Knative Serving                                 │  ← Serverless scaling
│  Dex (Authentication)                            │  ← User identity management
├─────────────────────────────────────────────────┤
│  Kubernetes Cluster                              │  ← Container orchestration
│  ┌──────┐ ┌──────┐ ┌──────┐ ┌──────┐            │
│  │Node 1│ │Node 2│ │Node 3│ │GPU   │            │
│  │ CPU  │ │ CPU  │ │ CPU  │ │Node 4│            │
│  └──────┘ └──────┘ └──────┘ └──────┘            │
└─────────────────────────────────────────────────┘

Key insight: Kubeflow does NOT replace K8s — it EXTENDS it with ML-specific tools.
You still need to understand K8s fundamentals (pods, deployments, services).
"""

print(kubeflow_architecture)

In [ ]:
# Cell 12 — Simulate Kubeflow pipeline submission
import json

kubeflow_pipeline = {
    "pipeline_name": "training-pipeline-v2",
    "namespace": "ml-team-alpha",
    "steps": [
        {"name": "data-validation", "image": "ml-service:v1.0", "resources": {"cpu": "2", "memory": "4Gi"}, "command": "python validate.py"},
        {"name": "feature-engineering", "image": "ml-service:v1.0", "resources": {"cpu": "4", "memory": "8Gi"}, "command": "python features.py"},
        {"name": "model-training", "image": "ml-service:v1.0", "resources": {"cpu": "8", "memory": "32Gi", "gpu": "1"}, "command": "python train.py"},
        {"name": "model-evaluation", "image": "ml-service:v1.0", "resources": {"cpu": "4", "memory": "8Gi"}, "command": "python evaluate.py"},
        {"name": "model-registration", "image": "ml-service:v1.0", "resources": {"cpu": "2", "memory": "4Gi"}, "command": "python register.py"},
    ],
}

print("🚀 Kubeflow Pipeline Definition:")
for i, step in enumerate(kubeflow_pipeline["steps"], 1):
    gpu = step['resources'].get('gpu', '0')
    gpu_str = f" 🎮 GPU: {gpu}" if gpu != '0' else ""
    print(f"   Step {i}: {step['name']} (CPU: {step['resources']['cpu']}, Mem: {step['resources']['memory']}{gpu_str})")
print(f"\n   Pipeline submitted to namespace: {kubeflow_pipeline['namespace']}")

---
## 6 - GitHub Actions: CI/CD for ML Projects

### Why CI/CD Matters for ML

In production ML, every code change should be automatically:
1. Linted and type-checked
2. Unit tested
3. Docker image built and pushed
4. Model smoke-tested

**GitHub Actions** is the most popular CI/CD platform. It runs workflows defined in `.github/workflows/` YAML files.

### Key Concepts

| Term | What It Means | Example |
|------|-------------|--------|
| **Workflow** | A YAML file defining automation | `.github/workflows/ml-ci.yml` |
| **Trigger** | What starts the workflow | `push to main`, `pull_request` |
| **Job** | A set of steps that run on one machine | `test`, `build`, `deploy` |
| **Step** | A single command or action | `pip install`, `pytest`, `docker build` |
| **Secret** | Encrypted env vars | `${{ secrets.DOCKER_PASSWORD }}` |


In [ ]:
# Cell 10 -- Generate a production GitHub Actions CI pipeline for ML

ci_yaml = '''name: ML Pipeline CI

on:
  push:
    branches: [main, staging]
  pull_request:
    branches: [main]

env:
  REGISTRY: ghcr.io
  IMAGE_NAME: ${{ github.repository }}/ml-service

jobs:
  lint-and-test:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: "3.11"
          cache: pip

      - name: Install dependencies
        run: pip install -r requirements.txt -r requirements-dev.txt

      - name: Lint (ruff is 10-100x faster than flake8)
        run: ruff check src/ tests/

      - name: Type check
        run: mypy src/ --ignore-missing-imports

      - name: Unit tests
        run: pytest tests/ -v --tb=short

  build-and-push:
    needs: lint-and-test
    runs-on: ubuntu-latest
    if: github.event_name == 'push'
    permissions:
      contents: read
      packages: write
    steps:
      - uses: actions/checkout@v4

      - name: Login to Container Registry
        uses: docker/login-action@v3
        with:
          registry: ${{ env.REGISTRY }}
          username: ${{ github.actor }}
          password: ${{ secrets.GITHUB_TOKEN }}

      - name: Build and push Docker image
        uses: docker/build-push-action@v5
        with:
          push: true
          tags: ${{ env.REGISTRY }}/${{ env.IMAGE_NAME }}:${{ github.sha }}
          cache-from: type=gha
          cache-to: type=gha,mode=max

  model-smoke-test:
    needs: build-and-push
    runs-on: ubuntu-latest
    steps:
      - name: Pull and test the image
        run: |
          docker pull ${{ env.REGISTRY }}/${{ env.IMAGE_NAME }}:${{ github.sha }}
          docker run --rm -d -p 8000:8000 --name test-server \\
            ${{ env.REGISTRY }}/${{ env.IMAGE_NAME }}:${{ github.sha }}
          sleep 10
          curl -f http://localhost:8000/health || exit 1
          docker stop test-server
'''

with open('.github_workflows_ml_ci.yml', 'w') as f:
    f.write(ci_yaml)

print('GitHub Actions CI Pipeline Generated')
print('\nPipeline stages:')
print('  1. lint-and-test  -> ruff + mypy + pytest')
print('  2. build-and-push -> Docker build with layer caching')
print('  3. smoke-test     -> Pull image, start server, hit /health')
print('\nKey best practices:')
print('  - Cache pip dependencies (faster builds)')
print('  - Use GitHub Container Registry (free for public repos)')
print('  - Docker layer caching via GitHub Actions cache')
print('  - Only build on push (not on PR -- save compute)')


---
### .dockerignore: Keep Your Images Clean

Just like `.gitignore` prevents files from entering your repo, `.dockerignore` prevents files from entering your Docker image.

**Why it matters:** Without `.dockerignore`, `docker build` copies your ENTIRE directory into the image -- including:
- `.git/` (hundreds of MB)
- `__pycache__/` (stale bytecode)
- `.env` files (SECRETS!)
- Model checkpoints (GB of weights)


In [ ]:
# Cell 11 -- Generate .dockerignore
dockerignore = """.git
.github
__pycache__
*.pyc
.env
.env.*
*.egg-info
dist/
build/
notebooks/
tests/
*.md
*.ipynb
models/checkpoints/
data/raw/
wandb/
.vscode/
"""

with open('.dockerignore', 'w') as f:
    f.write(dockerignore)

print('.dockerignore generated')
print('\nExcluded from Docker image:')
for line in dockerignore.strip().split('\n'):
    if line.strip():
        print(f'  {line}')
print('\nThis can reduce image build context from GB to MB.')


---
### Docker Compose: Multi-Container ML Systems

Real ML systems have MULTIPLE containers working together:
- **API server** -- serves predictions
- **Model server** -- runs vLLM for inference
- **Cache** -- Redis for response caching
- **Monitoring** -- Prometheus + Grafana

`docker-compose.yml` defines all containers and their connections in one file.


In [ ]:
# Cell 12 -- Generate docker-compose.yml for ML system
compose = '''version: "3.9"

services:
  # API Gateway -- receives requests, routes to model server
  api:
    build: .
    ports:
      - "8000:8000"
    environment:
      - MODEL_SERVER_URL=http://model:8080
      - REDIS_URL=redis://cache:6379
    depends_on:
      - model
      - cache
    restart: unless-stopped
    healthcheck:
      test: ["CMD", "curl", "-f", "http://localhost:8000/health"]
      interval: 30s
      timeout: 10s
      retries: 3

  # Model Server -- runs vLLM for high-throughput inference
  model:
    image: vllm/vllm-openai:latest
    deploy:
      resources:
        reservations:
          devices:
            - driver: nvidia
              count: 1
              capabilities: [gpu]
    environment:
      - MODEL_NAME=meta-llama/Llama-3-8B-Instruct
    ports:
      - "8080:8080"

  # Cache -- Redis for response caching (reduces GPU cost)
  cache:
    image: redis:7-alpine
    ports:
      - "6379:6379"
    volumes:
      - redis_data:/data

volumes:
  redis_data:
'''

with open('docker-compose.yml', 'w') as f:
    f.write(compose)

print('docker-compose.yml generated')
print('\nServices:')
print('  api:   FastAPI gateway        (port 8000)')
print('  model: vLLM inference server   (port 8080, GPU)')
print('  cache: Redis response cache    (port 6379)')
print('\nCommands:')
print('  docker compose up -d          # Start all services')
print('  docker compose logs -f api    # Follow API logs')
print('  docker compose down           # Stop everything')
print('  docker compose ps             # Check status')


---
## ✅ Knowledge Check

Test your understanding with these questions. Try to answer before expanding the details.

### Question 1: Kubernetes Limits
What happens if a Kubernetes pod exceeds its memory `limit`? What happens if it exceeds its CPU `limit`?

<details>
<summary>👉 Click to reveal answer</summary>

If a pod exceeds its **memory limit**, it is terminated immediately (`OOMKilled`) because memory cannot be compressed. If a pod exceeds its **CPU limit**, it is **throttled** (slowed down) but not killed, because CPU time is shareable over time.
</details>

### Question 2: Docker Multi-Stage Builds
Why shouldn't you just use a single large Dockerfile containing build tools (like `gcc`) for your production image?

<details>
<summary>👉 Click to reveal answer</summary>

1. **Security:** Build tools give attackers a larger surface area to exploit your container.
2. **Size:** A single-stage image can be 8GB+, making deployments very slow.
Multi-stage builds let you compile in one stage, and copy only the final artifacts to a minimal runtime image (~3GB).
</details>

### Question 3: Git Operations
Why is it considered bad practice to version control a 10GB dataset with standard `git`?

<details>
<summary>👉 Click to reveal answer</summary>

Git is optimized for text files and line-by-line diffs. Committing large binary files bloats the repository history permanently, causing every `git clone` to download gigabytes of data and eventually breaking Git. For data, you should use DVC (Data Version Control) or an object store like S3/lakeFS.
</details>


---
## 🔨 Practical Practice

Complete these automated exercises by writing the required definitions/commands.

### Exercise 1: Advanced Grep
Write a `grep` command to find all occurrences of either `ERROR` or `WARNING` in `training.log` and count the total number of matches.

### Exercise 2: Python to Dockerfile translation
Write the Dockerfile instructions to:
1. Start from `python:3.11-slim`
2. Install `transformers` and `torch` without creating cache files
3. Run the application `python app.py`

### Exercise 3: Kubernetes Scaling
Given a Kubernetes deployment named `model-server` in the `ml-prod` namespace, write the command to scale it to exactly 5 replicas.


In [ ]:
# ===== EXERCISE SOLUTIONS (try yourself first!) =====

print('Ex 1: grep -c "ERROR\\|WARNING" training.log')
print('      OR: grep -E -c "ERROR|WARNING" training.log')

print('\nEx 2:')
print('FROM python:3.11-slim')
print('RUN pip install --no-cache-dir transformers torch')
print('COPY app.py .')
print('CMD ["python", "app.py"]')

print('\nEx 3: kubectl scale deployment/model-server --replicas=5 -n ml-prod')


---

## 🎯 Summary & Key Takeaways

| Step | Tool | What You Learned | Why It Matters |
|------|------|-----------------|----------------|
| 1 | Linux CLI (grep/awk/sed) | Parse ML training logs to find OOM errors, NaN losses | Debug production failures remotely |
| 2 | Git | Version control everything in MLOps | Reproduce and rollback any experiment |
| 3 | Docker (multi-stage) | Package ML models with exact dependencies | "Works on my machine" is eliminated |
| 4 | Kubernetes | Orchestrate containers with GPU resource limits | Scale, self-heal, zero-downtime deploy |
| 5 | Kubeflow | ML-specific orchestration on K8s | Pipelines, tuning, serving — all integrated |

### 🧠 Key Questions

1. **"Do I really need Kubernetes for ML?"** → If you're deploying one model for 10 users, no. If you're serving multiple models to thousands of users with SLAs, absolutely yes.
2. **"Can I skip Docker and just install everything on the server?"** → You can, and you'll spend weeks debugging "dependency hell." Docker pays for itself on the first deployment.
3. **"Which orchestrator should I start with?"** → Metaflow if you're a data scientist. Kubeflow if your company already uses K8s. Airflow if you also do data engineering.

**Next →** `14_cloud_platforms_ai_infrastructure.ipynb` — Cloud Platforms & AI Infrastructure
